# WiDS Wildfire Survival — Breakthrough Metric-Aware Ensemble

In [1]:

import os, json, math, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.special import expit, logit
from scipy.optimize import minimize

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.isotonic import IsotonicRegression

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

RANDOM_STATE = 20260420
np.random.seed(RANDOM_STATE)
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")

ID_COL = "event_id"
TARGET_COLS = ["time_to_hit_hours", "event"]
HORIZONS = [12, 24, 48, 72]
PRED_COLS = [f"prob_{h}h" for h in HORIZONS]
REQ_COLS = [ID_COL] + PRED_COLS
EPS = 1e-8

def find_file(filename):
    roots = []
    if os.path.isdir("/kaggle/input"):
        roots.append("/kaggle/input")
    roots += [os.getcwd(), "/mnt/data"]
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cur, _, files in os.walk(root):
            if filename in files:
                hits.append(os.path.join(cur, filename))
    if not hits:
        raise FileNotFoundError(filename)
    hits = sorted(hits, key=lambda p: (("samplecsv" in p.lower()) or ("anonymous-contest" in p.lower()), len(p), p))
    return hits[0]

train = pd.read_csv(find_file("train.csv"))
test = pd.read_csv(find_file("test.csv"))
sample = pd.read_csv(find_file("sample_submission.csv"))

def sigmoid(x):
    return expit(np.clip(x, -40, 40))

def as_num(s, n, default=0.0):
    if s in n.columns:
        return pd.to_numeric(n[s], errors="coerce").to_numpy(dtype=float)
    return np.full(len(n), float(default), dtype=float)

def signed_log1p(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))

def put(df, name, values):
    df[name] = np.asarray(values, dtype=float)

def feature_engineer(df):
    out = df.copy()
    for c in out.columns:
        if c != ID_COL:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    n = len(out)
    dist = as_num("dist_min_ci_0_5h", out, np.nan)
    gap = np.maximum(dist - 5000.0, 0.0)
    dt = np.maximum(as_num("dt_first_last_0_5h", out, 5.0), 0.05)
    closing = as_num("closing_speed_m_per_h", out)
    abs_closing = as_num("closing_speed_abs_m_per_h", out)
    along = as_num("along_track_speed", out)
    slope = as_num("dist_slope_ci_0_5h", out)
    projected = as_num("projected_advance_m", out)
    accel = as_num("dist_accel_m_per_h2", out)
    radial = as_num("radial_growth_rate_m_per_h", out)
    centroid = as_num("centroid_speed_m_per_h", out)
    align = as_num("alignment_cos", out)
    align_abs = as_num("alignment_abs", out)
    cross = as_num("cross_track_component", out)
    area0 = as_num("area_first_ha", out)
    growth_abs = as_num("area_growth_abs_0_5h", out)
    growth_rate = as_num("area_growth_rate_ha_per_h", out)
    rel_growth = as_num("relative_growth_0_5h", out)
    r2 = as_num("dist_fit_r2_0_5h", out)

    speed_candidates = np.vstack([
        np.maximum(closing, 0),
        np.maximum(along, 0),
        np.maximum(-slope, 0),
        np.maximum(projected / dt, 0),
        np.maximum(radial, 0),
        np.maximum(centroid * np.maximum(align, 0), 0),
        np.maximum(abs_closing * np.maximum(align_abs, 0), 0),
        np.maximum((projected + 0.5 * np.maximum(accel, 0) * 25.0) / 5.0, 0),
    ])
    speed_candidates = np.where(np.isfinite(speed_candidates), speed_candidates, 0.0)
    speed_max = np.nanmax(speed_candidates, axis=0)
    speed_p70 = np.nanpercentile(speed_candidates, 70, axis=0)
    speed_mean = np.nanmean(speed_candidates, axis=0)

    put(out, "gap_to_5km_m", gap)
    put(out, "log1p_gap_to_5km", np.log1p(np.maximum(gap, 0)))
    put(out, "log1p_dist_min_ci", np.log1p(np.maximum(dist, 0)))
    put(out, "inv_dist_km_plus1", 1.0 / (1.0 + np.maximum(dist, 0) / 1000.0))
    put(out, "speed_max_toward_evac", speed_max)
    put(out, "speed_p70_toward_evac", speed_p70)
    put(out, "speed_mean_toward_evac", speed_mean)
    put(out, "closing_per_gap", closing / (gap + 500.0))
    put(out, "projected_per_gap", projected / (gap + 500.0))
    put(out, "radial_per_gap", radial / (gap + 500.0))
    put(out, "centroid_per_gap", centroid / (gap + 500.0))
    put(out, "closing_x_alignment", closing * align)
    put(out, "closing_x_abs_alignment", closing * align_abs)
    put(out, "along_x_alignment", along * align)
    put(out, "radial_x_alignment", radial * align)
    put(out, "growth_x_alignment", growth_rate * align_abs)
    put(out, "area_x_growth", np.log1p(np.maximum(area0, 0)) * np.log1p(np.maximum(growth_abs, 0)))
    put(out, "cross_track_abs", np.abs(cross))
    put(out, "cross_track_over_motion", np.abs(cross) / (np.abs(along) + np.abs(cross) + 1.0))
    put(out, "distance_trend_consistency", r2 * np.sign(np.maximum(closing, 0)))
    put(out, "temporal_density_0_5h", as_num("num_perimeters_0_5h", out) / (dt + 0.25))
    put(out, "already_within_5km", (dist <= 5000).astype(float))
    put(out, "within_10km", (dist <= 10000).astype(float))
    put(out, "within_20km", (dist <= 20000).astype(float))
    put(out, "far_50km_plus", (dist >= 50000).astype(float))
    put(out, "closing_positive", (closing > 0).astype(float))
    put(out, "aligned_and_closing", ((closing > 0) & (align > 0)).astype(float))
    put(out, "far_and_receding", ((dist > 30000) & (closing <= 0)).astype(float))

    for nm, sp in [
        ("max", speed_max),
        ("p70", speed_p70),
        ("mean", speed_mean),
        ("closing", np.maximum(closing, 0)),
        ("along", np.maximum(along, 0)),
    ]:
        eta = np.clip((gap + 250.0) / np.maximum(sp, 1.0), 0, 5000)
        put(out, f"eta_hours_{nm}", eta)
        put(out, f"inv_eta_{nm}", 1.0 / (1.0 + eta))

    for h in HORIZONS:
        m1 = (speed_max * h - gap) / (1300.0 + 0.22 * gap + 0.35 * speed_max * h)
        m2 = (speed_p70 * h - gap) / (1600.0 + 0.28 * gap + 0.30 * speed_p70 * h)
        m3 = (np.maximum(closing, 0) * h - gap) / (1800.0 + 0.33 * gap + 0.25 * np.maximum(closing, 0) * h)
        p = 0.48 * sigmoid(m1) + 0.32 * sigmoid(m2) + 0.20 * sigmoid(m3)
        p = np.where(dist <= 5000.0, np.maximum(p, 0.88 + 0.10 * (h >= 24)), p)
        put(out, f"phys_prob_{h}h", p)
        put(out, f"phys_logit_{h}h", logit(np.clip(p, 1e-5, 1 - 1e-5)))
        put(out, f"phys_margin_{h}h", speed_max * h - gap)
        put(out, f"phys_ratio_{h}h", (speed_max * h + 50.0) / (gap + 500.0))

    hour = as_num("event_start_hour", out)
    dow = as_num("event_start_dayofweek", out)
    mon = as_num("event_start_month", out)
    put(out, "hour_sin", np.sin(2 * np.pi * hour / 24.0))
    put(out, "hour_cos", np.cos(2 * np.pi * hour / 24.0))
    put(out, "dow_sin", np.sin(2 * np.pi * dow / 7.0))
    put(out, "dow_cos", np.cos(2 * np.pi * dow / 7.0))
    put(out, "month_sin", np.sin(2 * np.pi * (mon - 1) / 12.0))
    put(out, "month_cos", np.cos(2 * np.pi * (mon - 1) / 12.0))
    put(out, "peak_fire_season_flag", ((mon >= 6) & (mon <= 10)).astype(float))

    for c in list(out.columns):
        if c == ID_COL or c in TARGET_COLS:
            continue
        if any(k in c for k in ["growth", "speed", "slope", "change", "accel", "advance", "component", "displacement", "rate"]):
            vals = pd.to_numeric(out[c], errors="coerce").to_numpy(dtype=float)
            if np.isfinite(vals).any() and np.nanmax(np.abs(vals)) > 10:
                put(out, f"slog1p_{c}", signed_log1p(vals))

    pairs = [
        ("log1p_dist_min_ci", "speed_max_toward_evac"),
        ("log1p_gap_to_5km", "closing_speed_m_per_h"),
        ("log1p_gap_to_5km", "alignment_abs"),
        ("inv_dist_km_plus1", "speed_max_toward_evac"),
        ("inv_dist_km_plus1", "area_growth_rate_ha_per_h"),
        ("phys_prob_24h", "alignment_abs"),
        ("phys_prob_48h", "alignment_abs"),
        ("eta_hours_max", "alignment_abs"),
        ("eta_hours_max", "low_temporal_resolution_0_5h"),
    ]
    for a, b in pairs:
        if a in out.columns and b in out.columns:
            put(out, f"{a}__x__{b}", pd.to_numeric(out[a], errors="coerce").to_numpy(float) * pd.to_numeric(out[b], errors="coerce").to_numpy(float))
    return out

train_fe = feature_engineer(train)
test_fe = feature_engineer(test)

base_features = [c for c in train_fe.columns if c not in [ID_COL] + TARGET_COLS]
for c in base_features:
    if c not in test_fe.columns:
        test_fe[c] = np.nan

Xdf = train_fe[base_features].replace([np.inf, -np.inf], np.nan)
XTdf = test_fe[base_features].replace([np.inf, -np.inf], np.nan)

valid = []
for c in base_features:
    s = Xdf[c]
    if s.notna().sum() > 0 and s.nunique(dropna=True) > 1:
        valid.append(c)
Xdf = Xdf[valid]
XTdf = XTdf[valid]

imputer = SimpleImputer(strategy="median")
Xn = imputer.fit_transform(Xdf)
Xtn = imputer.transform(XTdf)

lo = np.nanpercentile(Xn, 0.5, axis=0)
hi = np.nanpercentile(Xn, 99.5, axis=0)
Xn = np.clip(Xn, lo, hi)
Xtn = np.clip(Xtn, lo, hi)

scaler = RobustScaler(quantile_range=(10, 90))
Xs = scaler.fit_transform(Xn)
Xts = scaler.transform(Xtn)

n_train = len(train)
n_test = len(test)
time_y = pd.to_numeric(train["time_to_hit_hours"], errors="coerce").fillna(72).to_numpy(float)
event_y = pd.to_numeric(train["event"], errors="coerce").fillna(0).astype(int).to_numpy()

def horizon_target(h):
    observed = (event_y == 1) | (time_y >= h)
    y = ((event_y == 1) & (time_y <= h)).astype(int)
    return y, observed

def brier(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean((y - p) ** 2))

def splits_for(y, seeds=(13, 37), max_splits=5):
    y = np.asarray(y).astype(int)
    n = len(y)
    if n < 4:
        return []
    vals, counts = np.unique(y, return_counts=True)
    out = []
    if len(vals) == 2 and counts.min() >= 2:
        ns = max(2, min(max_splits, int(counts.min())))
        for s in seeds:
            out += list(StratifiedKFold(n_splits=ns, shuffle=True, random_state=s).split(np.zeros(n), y))
    else:
        ns = max(2, min(max_splits, n))
        for s in seeds:
            out += list(KFold(n_splits=ns, shuffle=True, random_state=s).split(np.zeros(n)))
    return out

def balanced_weights(y):
    y = np.asarray(y).astype(int)
    vals, counts = np.unique(y, return_counts=True)
    w = np.ones(len(y), dtype=float)
    for v, c in zip(vals, counts):
        w[y == v] = len(y) / (len(vals) * max(c, 1))
    return w

def predict_prob(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        if getattr(p, "ndim", 1) == 2 and p.shape[1] >= 2:
            return p[:, 1]
        return np.asarray(p).ravel()
    if hasattr(model, "decision_function"):
        return sigmoid(model.decision_function(X))
    return np.asarray(model.predict(X), dtype=float).ravel()

def safe_fit_predict(factory, use_scaled, tr_idx, va_idx, y_full):
    XA = Xs if use_scaled else Xn
    XTA = Xts if use_scaled else Xtn
    ytr = y_full[tr_idx]
    if len(np.unique(ytr)) < 2:
        c = float(np.mean(ytr))
        return np.full(len(va_idx), c), np.full(n_test, c)
    model = factory()
    sw = balanced_weights(ytr)
    try:
        model.fit(XA[tr_idx], ytr, sample_weight=sw)
    except Exception:
        try:
            model.fit(XA[tr_idx], ytr)
        except Exception:
            c = float(np.mean(ytr))
            return np.full(len(va_idx), c), np.full(n_test, c)
    return np.clip(predict_prob(model, XA[va_idx]), 0, 1), np.clip(predict_prob(model, XTA), 0, 1)

cands = {h: {"names": [], "oof": [], "test": []} for h in HORIZONS}

def add_cand(h, name, oof, testp):
    oof = np.asarray(oof, dtype=float).reshape(-1)
    testp = np.asarray(testp, dtype=float).reshape(-1)
    if len(oof) != n_train or len(testp) != n_test:
        return
    fallback = np.nanmean(oof) if np.isfinite(oof).any() else 0.5
    oof = np.nan_to_num(oof, nan=fallback, posinf=fallback, neginf=fallback)
    testp = np.nan_to_num(testp, nan=fallback, posinf=fallback, neginf=fallback)
    cands[h]["names"].append(name)
    cands[h]["oof"].append(np.clip(oof, 0, 1))
    cands[h]["test"].append(np.clip(testp, 0, 1))

for h in HORIZONS:
    y_h, m_h = horizon_target(h)
    prior = float(np.mean(y_h[m_h])) if m_h.any() else float(np.mean(event_y))
    add_cand(h, f"prior_{h}", np.full(n_train, prior), np.full(n_test, prior))
    col = f"phys_prob_{h}h"
    if col in train_fe.columns and col in test_fe.columns:
        add_cand(h, f"physics_{h}", train_fe[col].to_numpy(float), test_fe[col].to_numpy(float))

# Local nearest-neighbor empirical estimates.
try:
    Dtr = pairwise_distances(Xs, Xs)
    np.fill_diagonal(Dtr, np.inf)
    Dte = pairwise_distances(Xts, Xs)
    for h in HORIZONS:
        y_h, m_h = horizon_target(h)
        obs = np.where(m_h)[0]
        if len(obs) < 8 or len(np.unique(y_h[obs])) < 2:
            continue
        prior = float(np.mean(y_h[obs]))
        dtr = Dtr[:, obs]
        dte = Dte[:, obs]
        for k in [7, 11, 17, 29, min(55, len(obs))]:
            k = int(min(k, len(obs)))
            if k < 3:
                continue
            for pow_ in [1.0, 2.0]:
                nb = np.argpartition(dtr, kth=k - 1, axis=1)[:, :k]
                ds = np.take_along_axis(dtr, nb, axis=1)
                ys = y_h[obs][nb]
                w = np.where(np.isfinite(ds), 1.0 / np.maximum(ds, 1e-6) ** pow_, 0.0)
                sm = 1.5 + 0.04 * k
                po = (np.sum(w * ys, axis=1) + sm * prior) / (np.sum(w, axis=1) + sm)
                nbt = np.argpartition(dte, kth=k - 1, axis=1)[:, :k]
                dst = np.take_along_axis(dte, nbt, axis=1)
                yst = y_h[obs][nbt]
                wt = 1.0 / np.maximum(dst, 1e-6) ** pow_
                pt = (np.sum(wt * yst, axis=1) + sm * prior) / (np.sum(wt, axis=1) + sm)
                add_cand(h, f"nn_k{k}_p{pow_}_{h}", po, pt)
except Exception as e:
    print("NN skipped:", repr(e))

def model_specs(seed):
    specs = [
        ("log_c025", True, lambda s=seed: LogisticRegression(C=0.25, penalty="l2", solver="lbfgs", max_iter=5000, class_weight="balanced", random_state=s)),
        ("log_c1", True, lambda s=seed: LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=5000, class_weight="balanced", random_state=s)),
        ("log_l1", True, lambda s=seed: LogisticRegression(C=0.50, penalty="l1", solver="liblinear", max_iter=5000, class_weight="balanced", random_state=s)),
        ("hgb_s", False, lambda s=seed: HistGradientBoostingClassifier(max_iter=130, learning_rate=0.035, max_leaf_nodes=5, min_samples_leaf=12, l2_regularization=0.20, random_state=s)),
        ("hgb_m", False, lambda s=seed: HistGradientBoostingClassifier(max_iter=170, learning_rate=0.028, max_leaf_nodes=7, min_samples_leaf=9, l2_regularization=0.08, random_state=s)),
        ("gb1", False, lambda s=seed: GradientBoostingClassifier(n_estimators=150, learning_rate=0.030, max_depth=1, subsample=0.85, min_samples_leaf=8, random_state=s)),
        ("gb2", False, lambda s=seed: GradientBoostingClassifier(n_estimators=140, learning_rate=0.030, max_depth=2, subsample=0.80, min_samples_leaf=8, random_state=s)),
        ("et_deep", False, lambda s=seed: ExtraTreesClassifier(n_estimators=350, min_samples_leaf=3, max_features=0.55, class_weight="balanced_subsample", random_state=s, n_jobs=-1)),
        ("et_smooth", False, lambda s=seed: ExtraTreesClassifier(n_estimators=420, max_depth=5, min_samples_leaf=4, max_features=0.70, bootstrap=True, class_weight="balanced_subsample", random_state=s, n_jobs=-1)),
        ("rf", False, lambda s=seed: RandomForestClassifier(n_estimators=320, max_depth=6, min_samples_leaf=4, max_features=0.65, class_weight="balanced_subsample", random_state=s, n_jobs=-1)),
        ("ada", False, lambda s=seed: AdaBoostClassifier(n_estimators=110, learning_rate=0.045, random_state=s)),
        ("knn", True, lambda s=seed: KNeighborsClassifier(n_neighbors=11, weights="distance")),
        ("gnb", True, lambda s=seed: GaussianNB(var_smoothing=1e-8)),
    ]
    if XGBClassifier is not None:
        specs += [
            ("xgb2", False, lambda s=seed: XGBClassifier(n_estimators=170, max_depth=2, learning_rate=0.030, subsample=0.90, colsample_bytree=0.70, min_child_weight=3, reg_lambda=5.0, reg_alpha=0.10, objective="binary:logistic", eval_metric="logloss", random_state=s, n_jobs=1, verbosity=0)),
            ("xgb3", False, lambda s=seed: XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.025, subsample=0.82, colsample_bytree=0.65, min_child_weight=4, reg_lambda=8.0, reg_alpha=0.20, objective="binary:logistic", eval_metric="logloss", random_state=s, n_jobs=1, verbosity=0)),
        ]
    if LGBMClassifier is not None:
        specs += [
            ("lgb", False, lambda s=seed: LGBMClassifier(n_estimators=180, learning_rate=0.022, num_leaves=7, max_depth=3, min_child_samples=10, subsample=0.85, colsample_bytree=0.65, reg_lambda=6.0, reg_alpha=0.05, objective="binary", class_weight="balanced", random_state=s, n_jobs=1, verbose=-1)),
        ]
    if CatBoostClassifier is not None:
        specs += [
            ("cat", False, lambda s=seed: CatBoostClassifier(iterations=180, depth=2, learning_rate=0.030, l2_leaf_reg=10, loss_function="Logloss", random_seed=s, verbose=False, allow_writing_files=False)),
        ]
    return specs

for h in [12, 24, 48]:
    y_h, m_h = horizon_target(h)
    obs = np.where(m_h)[0]
    if len(obs) < 10 or len(np.unique(y_h[obs])) < 2:
        continue
    sp = splits_for(y_h[obs], seeds=(17, 43), max_splits=5)
    for seed in [101, 211]:
        for name, use_scaled, factory in model_specs(seed):
            oof = np.full(n_train, np.nan)
            oof_count = np.zeros(n_train)
            test_sum = np.zeros(n_test)
            cnt = 0
            for tr_rel, va_rel in sp:
                tr_idx, va_idx = obs[tr_rel], obs[va_rel]
                pva, pte = safe_fit_predict(factory, use_scaled, tr_idx, va_idx, y_h)
                oof[va_idx] = np.where(np.isfinite(oof[va_idx]), (oof[va_idx] * oof_count[va_idx] + pva) / (oof_count[va_idx] + 1), pva)
                oof_count[va_idx] += 1
                test_sum += pte
                cnt += 1
            if cnt:
                prior = float(np.mean(y_h[obs]))
                oof[~np.isfinite(oof)] = prior
                add_cand(h, f"direct_{name}_s{seed}_{h}", oof, test_sum / cnt)
            gc.collect()

# Discrete-time hazard stack.
INTERVALS = [(0, 12), (12, 24), (24, 48), (48, 72)]

def expand_hazard(indices, XA):
    rows, ys = [], []
    for i in indices:
        t, e = time_y[i], event_y[i]
        base = XA[i]
        for k, (s, en) in enumerate(INTERVALS):
            if e == 1:
                if t <= s:
                    break
                y = int(t <= en)
            else:
                if t >= en:
                    y = 0
                else:
                    break
            rows.append(np.r_[base, [k, s/72, en/72, (en-s)/72, en == 12, en == 24, en == 48, en == 72]])
            ys.append(y)
            if e == 1 and t <= en:
                break
    if not rows:
        return np.empty((0, XA.shape[1] + 8)), np.array([], dtype=int)
    return np.vstack(rows), np.asarray(ys, dtype=int)

def pred_hazard(model, XA):
    rows = []
    for i in range(len(XA)):
        base = XA[i]
        for k, (s, en) in enumerate(INTERVALS):
            rows.append(np.r_[base, [k, s/72, en/72, (en-s)/72, en == 12, en == 24, en == 48, en == 72]])
    haz = np.clip(predict_prob(model, np.vstack(rows)).reshape(len(XA), 4), 1e-5, 1 - 1e-5)
    return 1.0 - np.cumprod(1.0 - haz, axis=1)

def hazard_specs(seed):
    specs = [
        ("hz_log", True, lambda s=seed: LogisticRegression(C=0.45, solver="lbfgs", max_iter=5000, class_weight="balanced", random_state=s)),
        ("hz_hgb", False, lambda s=seed: HistGradientBoostingClassifier(max_iter=170, learning_rate=0.030, max_leaf_nodes=7, min_samples_leaf=14, l2_regularization=0.15, random_state=s)),
        ("hz_et", False, lambda s=seed: ExtraTreesClassifier(n_estimators=350, max_depth=6, min_samples_leaf=6, max_features=0.65, class_weight="balanced_subsample", random_state=s, n_jobs=-1)),
    ]
    if XGBClassifier is not None:
        specs.append(("hz_xgb", False, lambda s=seed: XGBClassifier(n_estimators=160, max_depth=2, learning_rate=0.030, subsample=0.85, colsample_bytree=0.70, min_child_weight=4, reg_lambda=8.0, reg_alpha=0.20, objective="binary:logistic", eval_metric="logloss", random_state=s, n_jobs=1, verbosity=0)))
    if LGBMClassifier is not None:
        specs.append(("hz_lgb", False, lambda s=seed: LGBMClassifier(n_estimators=170, learning_rate=0.022, num_leaves=7, max_depth=3, min_child_samples=14, subsample=0.85, colsample_bytree=0.70, reg_lambda=8.0, objective="binary", class_weight="balanced", random_state=s, n_jobs=1, verbose=-1)))
    return specs

try:
    row_splits = splits_for(event_y, seeds=(23, 61), max_splits=5)
    for seed in [503, 907]:
        for name, use_scaled, factory in hazard_specs(seed):
            XA, XTA = (Xs, Xts) if use_scaled else (Xn, Xtn)
            oof = np.full((n_train, 4), np.nan)
            test_sum = np.zeros((n_test, 4))
            cnt = 0
            for tr_idx, va_idx in row_splits:
                Xh, yh = expand_hazard(tr_idx, XA)
                if len(np.unique(yh)) < 2:
                    continue
                model = factory()
                try:
                    model.fit(Xh, yh, sample_weight=balanced_weights(yh))
                except Exception:
                    model.fit(Xh, yh)
                oof[va_idx] = pred_hazard(model, XA[va_idx])
                test_sum += pred_hazard(model, XTA)
                cnt += 1
            if cnt:
                testp = test_sum / cnt
                for j, h in enumerate(HORIZONS):
                    y_h, m_h = horizon_target(h)
                    prior = float(np.mean(y_h[m_h])) if m_h.any() else float(np.mean(event_y))
                    oo = oof[:, j]
                    oo[~np.isfinite(oo)] = prior
                    add_cand(h, f"hazard_{name}_s{seed}_{h}", oo, testp[:, j])
except Exception as e:
    print("Hazard skipped:", repr(e))

# Penalized CoxPH with Breslow baseline.
def fit_cox(X, t, e, alpha=1.0, maxiter=130):
    X = np.asarray(X, dtype=float)
    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)
    ev = np.where(e == 1)[0]
    if len(ev) < 4:
        raise ValueError("too few events")
    mu, sd = X.mean(axis=0), X.std(axis=0) + 1e-6
    Z = (X - mu) / sd
    keep = np.where(Z.std(axis=0) > 1e-7)[0]
    Z = Z[:, keep]
    ev = np.where(e == 1)[0]
    risk = (t[None, :] >= t[ev, None]).astype(float)
    Zev = Z[ev]
    m, p = len(ev), Z.shape[1]
    def obj(beta):
        eta = np.clip(Z @ beta, -35, 35)
        ex = np.exp(eta)
        den = risk @ ex + 1e-12
        loss = -(eta[ev].sum() - np.log(den).sum()) / m + 0.5 * alpha * np.sum(beta * beta) / max(p, 1)
        wx = risk * ex[None, :]
        meanx = wx @ Z / den[:, None]
        grad = -(Zev - meanx).sum(axis=0) / m + alpha * beta / max(p, 1)
        return loss, grad
    res = minimize(lambda b: obj(b), np.zeros(p), jac=True, method="L-BFGS-B", options={"maxiter": maxiter, "ftol": 1e-8})
    beta = res.x if np.isfinite(res.x).all() else np.zeros(p)
    eta = np.clip(Z @ beta, -35, 35)
    ex = np.exp(eta)
    steps = []
    ch = 0.0
    for ut in np.sort(np.unique(t[e == 1])):
        d = np.sum((t == ut) & (e == 1))
        den = np.sum(ex[t >= ut]) + 1e-12
        ch += d / den
        steps.append((float(ut), float(ch)))
    return {"mu": mu, "sd": sd, "keep": keep, "beta": beta, "steps": steps}

def pred_cox(mod, X):
    Z = (np.asarray(X, dtype=float) - mod["mu"]) / mod["sd"]
    Z = Z[:, mod["keep"]]
    rr = np.exp(np.clip(Z @ mod["beta"], -35, 35))
    out = np.zeros((len(X), 4))
    for j, h in enumerate(HORIZONS):
        h0 = 0.0
        for ut, ch in mod["steps"]:
            if ut <= h:
                h0 = ch
            else:
                break
        out[:, j] = 1.0 - np.exp(-h0 * rr)
    return np.clip(out, 0, 1)

try:
    cox_splits = splits_for(event_y, seeds=(29, 79), max_splits=5)
    for alpha in [0.08, 0.25, 0.75, 2.0, 6.0]:
        oof = np.full((n_train, 4), np.nan)
        test_sum = np.zeros((n_test, 4))
        cnt = 0
        for tr_idx, va_idx in cox_splits:
            try:
                mod = fit_cox(Xs[tr_idx], time_y[tr_idx], event_y[tr_idx], alpha=alpha)
                oof[va_idx] = pred_cox(mod, Xs[va_idx])
                test_sum += pred_cox(mod, Xts)
                cnt += 1
            except Exception:
                continue
        if cnt:
            testp = test_sum / cnt
            for j, h in enumerate(HORIZONS):
                y_h, m_h = horizon_target(h)
                prior = float(np.mean(y_h[m_h])) if m_h.any() else float(np.mean(event_y))
                oo = oof[:, j]
                oo[~np.isfinite(oo)] = prior
                add_cand(h, f"cox_a{alpha}_{h}", oo, testp[:, j])
except Exception as e:
    print("Cox skipped:", repr(e))

def calibrations(po, pt, y):
    po = np.clip(po, 1e-5, 1 - 1e-5)
    pt = np.clip(pt, 1e-5, 1 - 1e-5)
    y = np.asarray(y, dtype=int)
    base = float(np.mean(y))
    out = [("raw", po, pt)]
    for lam in [0.03, 0.06, 0.10, 0.16, 0.24]:
        out.append((f"shrink{lam}", (1-lam)*po + lam*base, (1-lam)*pt + lam*base))
    L = logit(po).reshape(-1, 1)
    Lt = logit(pt).reshape(-1, 1)
    for C in [0.08, 0.2, 0.5, 1.0]:
        try:
            lr = LogisticRegression(C=C, solver="lbfgs", max_iter=2000)
            lr.fit(L, y)
            co = lr.predict_proba(L)[:, 1]
            ct = lr.predict_proba(Lt)[:, 1]
            for a in [0.45, 0.65, 0.85]:
                out.append((f"platt{C}_{a}", a*co + (1-a)*po, a*ct + (1-a)*pt))
        except Exception:
            pass
    try:
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
        iso.fit(po, y)
        co = iso.predict(po)
        ct = iso.predict(pt)
        for a in [0.35, 0.55, 0.75]:
            out.append((f"iso{a}", a*co + (1-a)*po, a*ct + (1-a)*pt))
    except Exception:
        pass
    return [(n, np.clip(a, 0, 1), np.clip(b, 0, 1)) for n, a, b in out]

def greedy_blend(P, T, y, topn=30, iters=110):
    P = np.asarray(P, dtype=float)
    T = np.asarray(T, dtype=float)
    y = np.asarray(y, dtype=int)
    finite = np.isfinite(P).all(axis=0) & np.isfinite(T).all(axis=0)
    P, T = P[:, finite], T[:, finite]
    if P.shape[1] == 0:
        base = float(np.mean(y))
        return np.full(len(y), base), np.full(n_test, base), {"fallback": base}
    keep = []
    seen = set()
    for j in range(P.shape[1]):
        if np.std(P[:, j]) < 1e-9:
            continue
        key = tuple(np.round(P[:, j], 6))
        if key in seen:
            continue
        seen.add(key)
        keep.append(j)
    if not keep:
        base = float(np.mean(y))
        return np.full(len(y), base), np.full(n_test, base), {"fallback": base}
    P, T = np.clip(P[:, keep], 1e-5, 1-1e-5), np.clip(T[:, keep], 1e-5, 1-1e-5)
    scores = np.array([brier(y, P[:, j]) for j in range(P.shape[1])])
    top = np.argsort(scores)[:min(topn, P.shape[1])]
    Pk, Tk = P[:, top], T[:, top]
    # add mean, median, and trimmed mean consensus
    P_aug = [Pk, Pk.mean(axis=1, keepdims=True), np.median(Pk, axis=1, keepdims=True)]
    T_aug = [Tk, Tk.mean(axis=1, keepdims=True), np.median(Tk, axis=1, keepdims=True)]
    if Pk.shape[1] >= 5:
        lo = np.percentile(Pk, 15, axis=1)
        hi = np.percentile(Pk, 85, axis=1)
        trim = np.array([row[(row >= lo[i]) & (row <= hi[i])].mean() for i, row in enumerate(Pk)])
        trimt = np.array([row[(row >= np.percentile(row, 15)) & (row <= np.percentile(row, 85))].mean() for row in Tk])
        P_aug.append(trim.reshape(-1, 1))
        T_aug.append(trimt.reshape(-1, 1))
    P_aug = np.column_stack(P_aug)
    T_aug = np.column_stack(T_aug)
    best_j = int(np.argmin([brier(y, P_aug[:, j]) for j in range(P_aug.shape[1])]))
    blend = P_aug[:, best_j].copy()
    blend_t = T_aug[:, best_j].copy()
    best = brier(y, blend)
    path = [(best_j, 1.0, best)]
    for _ in range(iters):
        improved = False
        best_tuple = None
        for j in range(P_aug.shape[1]):
            for a in [0.015, 0.025, 0.04, 0.06, 0.085, 0.12, 0.18, 0.26, 0.38, 0.52]:
                c = (1-a)*blend + a*P_aug[:, j]
                s = brier(y, c)
                if s + 1e-12 < best:
                    best = s
                    best_tuple = (j, a, c, (1-a)*blend_t + a*T_aug[:, j], s)
                    improved = True
        if not improved:
            break
        j, a, blend, blend_t, best = best_tuple
        path.append((int(j), float(a), float(best)))
    best_name, best_o, best_t, best_s = "raw", blend, blend_t, brier(y, blend)
    for name, co, ct in calibrations(blend, blend_t, y):
        s = brier(y, co)
        margin = 0.0 if name.startswith("shrink") or name == "raw" else -2e-5
        if s < best_s + margin:
            best_name, best_o, best_t, best_s = name, co, ct, s
    return np.clip(best_o, 0, 1), np.clip(best_t, 0, 1), {"n_candidates": int(P.shape[1]), "path_len": len(path), "raw_brier": float(brier(y, blend)), "best_brier": float(best_s), "calibration": best_name}

final = np.zeros((n_test, 4))
report = {}
for j, h in enumerate(HORIZONS):
    y_h, m_h = horizon_target(h)
    obs = np.where(m_h)[0]
    prior = float(np.mean(y_h[obs])) if len(obs) else float(np.mean(event_y))
    if len(obs) < 5 or len(np.unique(y_h[obs])) < 2 or not cands[h]["oof"]:
        final[:, j] = prior
        report[h] = {"fallback_prior": prior}
        continue
    P = np.column_stack(cands[h]["oof"])
    T = np.column_stack(cands[h]["test"])
    # Rank-smoothed variants help urgency ordering while keeping calibration anchored.
    extras_p, extras_t = [], []
    for c in range(P.shape[1]):
        try:
            rtr = pd.Series(P[:, c]).rank(pct=True).to_numpy()
            sv = np.sort(P[:, c])
            rte = np.searchsorted(sv, T[:, c], side="right") / max(len(sv), 1)
            extras_p.append(0.65 * P[:, c] + 0.35 * (0.15 + 0.70 * rtr))
            extras_t.append(0.65 * T[:, c] + 0.35 * (0.15 + 0.70 * rte))
        except Exception:
            pass
    if extras_p:
        P = np.column_stack([P] + extras_p)
        T = np.column_stack([T] + extras_t)
    po, pt, rep = greedy_blend(P[obs], T, y_h[obs], topn=32, iters=120)
    final[:, j] = pt
    rep.update({"prior": prior, "observed_n": int(len(obs)), "positive_n": int(y_h[obs].sum()), "raw_candidate_count": len(cands[h]["names"])})
    report[h] = rep

final = np.clip(final, 0, 1)
if "dist_min_ci_0_5h" in test.columns:
    d = pd.to_numeric(test["dist_min_ci_0_5h"], errors="coerce").to_numpy(float)
    close0 = d <= 5000
    final[close0, 0] = np.maximum(final[close0, 0], 0.90)
    final[close0, 1] = np.maximum(final[close0, 1], 0.96)
    final[close0, 2] = np.maximum(final[close0, 2], 0.985)
    far = d >= 75000
    final[far, 0] = np.minimum(final[far, 0], 0.18)
    final[far, 1] = np.minimum(final[far, 1], 0.32)

final[:, :3] = np.maximum.accumulate(final[:, :3], axis=1)
final[:, 3] = 1.0
final = np.maximum.accumulate(final, axis=1)
final = np.clip(final, 0, 1)

pred = pd.DataFrame({ID_COL: test[ID_COL].values})
for j, c in enumerate(PRED_COLS):
    pred[c] = final[:, j]

submission = sample[[ID_COL]].merge(pred, on=ID_COL, how="left")
for j, h in enumerate(HORIZONS):
    c = PRED_COLS[j]
    if submission[c].isna().any():
        y_h, m_h = horizon_target(h)
        prior = float(np.mean(y_h[m_h])) if m_h.any() else float(np.mean(event_y))
        submission[c] = submission[c].fillna(1.0 if h == 72 else prior)

submission = submission[REQ_COLS]
submission[PRED_COLS] = submission[PRED_COLS].clip(0, 1)
submission[PRED_COLS] = np.maximum.accumulate(submission[PRED_COLS].to_numpy(float), axis=1)
submission["prob_72h"] = 1.0

assert list(submission.columns) == REQ_COLS
assert len(submission) == len(sample)
assert submission[ID_COL].is_unique
assert set(submission[ID_COL]) == set(sample[ID_COL])
assert submission[PRED_COLS].notna().all().all()
assert ((submission[PRED_COLS] >= 0) & (submission[PRED_COLS] <= 1)).all().all()
assert (np.diff(submission[PRED_COLS].to_numpy(float), axis=1) >= -1e-12).all()

submission.to_csv(OUTPUT_PATH, index=False)
try:
    with open(os.path.join(WORK_DIR, "blend_report.json"), "w") as f:
        json.dump(report, f, indent=2, default=str)
except Exception:
    pass

print("Saved:", OUTPUT_PATH)
print(submission.head())
print(submission[PRED_COLS].describe())
for h in HORIZONS:
    print("H", h, "candidates", len(cands[h]["names"]), report.get(h, {}))


Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.001469  0.002584  0.002584       1.0
1  13353600  0.900000  0.977837  0.996760       1.0
2  13942327  0.001451  0.002043  0.002043       1.0
3  16112781  0.900000  0.977398  0.996345       1.0
4  17132808  0.013434  0.013434  0.013434       1.0
        prob_12h   prob_24h   prob_48h  prob_72h
count  95.000000  95.000000  95.000000      95.0
mean    0.270340   0.288781   0.298373       1.0
std     0.414021   0.441709   0.452241       0.0
min     0.001382   0.001455   0.002031       1.0
25%     0.001521   0.002240   0.002781       1.0
50%     0.001615   0.002613   0.002918       1.0
75%     0.900000   0.960000   0.988601       1.0
max     0.997265   0.997265   0.997708       1.0
H 12 candidates 61 {'n_candidates': 102, 'path_len': 70, 'raw_brier': 0.05313517831778305, 'best_brier': 0.047161153415334466, 'calibration': 'iso0.75', 'prior': 0.22790697674418606, 'observed_n': 215, 'positi